In [11]:
import sys
sys.path.insert(0, './tools')
from tecplot_lazy_reader import TecplotLazyReader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Initialize reader
reader = TecplotLazyReader('./flood-log-7/tecplot-output.dat')
reader.parse_header()

print("File metadata:")
print(f"  Variables: {reader.variables}")
print(f"  Grid size: I={reader.zone['I']}, J={reader.zone['J']}, K={reader.zone['K']}")
print(f"  Total points: {reader.zone['I'] * reader.zone['J'] * reader.zone['K']:,}")
print(f"  Data packing: {reader.datapacking}")

File metadata:
  Variables: ['X', 'Y', 'Z', 'U', 'V', 'W', 'uu', 'vv', 'ww', 'uv', 'vw', 'uw', 'K', 'Nv']
  Grid size: I=1325, J=73, K=1325
  Total points: 128,160,625
  Data packing: BLOCK


In [18]:
# Example 4: Advanced post-processing - Line probes, transverse analysis

# Extract a line probe (cross-section at a fixed X)
x_target_idx = 200  # Index in I direction
line_probe = df[df['X'].round(4) == df.iloc[x_target_idx * J]['X']]  # Get points at same X

if len(line_probe) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle(f'Line Probe Analysis at X ≈ {line_probe.iloc[0]["X"]:.3f}', fontweight='bold')
    
    # U profile
    axes[0].plot(line_probe['Y'], line_probe['U'], 'o-', label='U')
    axes[0].set_xlabel('Y')
    axes[0].set_ylabel('U velocity (m/s)')
    axes[0].set_title('U Profile')
    axes[0].grid(True, alpha=0.3)
    
    # V profile  
    axes[1].plot(line_probe['Y'], line_probe['V'], 's-', label='V', color='orange')
    axes[1].set_xlabel('Y')
    axes[1].set_ylabel('V velocity (m/s)')
    axes[1].set_title('V Profile')
    axes[1].grid(True, alpha=0.3)
    
    # Velocity magnitude profile
    vel_mag_probe = np.sqrt(line_probe['U']**2 + line_probe['V']**2 + line_probe['W']**2)
    axes[2].plot(line_probe['Y'], vel_mag_probe, '^-', label='|V|', color='green')
    axes[2].set_xlabel('Y')
    axes[2].set_ylabel('Velocity magnitude (m/s)')
    axes[2].set_title('|V| Profile')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No exact X match found for line probe extraction")

print(f"\n--- Post-Processing Options Summary ---")
print(f"1. Extract different 2D planes:")
print(f"   - XY plane: reader.extract_2d_slice(plane='XY', index=z_idx)")
print(f"   - XZ plane: reader.extract_2d_slice(plane='XZ', index=y_idx)")
print(f"   - YZ plane: reader.extract_2d_slice(plane='YZ', index=x_idx)")
print(f"\n2. Extract variables of interest:")
print(f"   - Variables available: {reader.variables}")
print(f"   - Pass var_indices=[...] to extract_2d_slice()")
print(f"\n3. Save to different formats:")
print(f"   - CSV (default): output_csv='file.csv'")
print(f"   - Convert to HDF5 or Zarr for efficient slicing")
print(f"\n4. Create visualizations:")
print(f"   - Contour plots (as shown above)")
print(f"   - Vector fields (quiver plots)")
print(f"   - Line probes for profile analysis")
print(f"   - Streamlines for flow visualization")

No exact X match found for line probe extraction

--- Post-Processing Options Summary ---
1. Extract different 2D planes:
   - XY plane: reader.extract_2d_slice(plane='XY', index=z_idx)
   - XZ plane: reader.extract_2d_slice(plane='XZ', index=y_idx)
   - YZ plane: reader.extract_2d_slice(plane='YZ', index=x_idx)

2. Extract variables of interest:
   - Variables available: ['X', 'Y', 'Z', 'U', 'V', 'W', 'uu', 'vv', 'ww', 'uv', 'vw', 'uw', 'K', 'Nv']
   - Pass var_indices=[...] to extract_2d_slice()

3. Save to different formats:
   - CSV (default): output_csv='file.csv'
   - Convert to HDF5 or Zarr for efficient slicing

4. Create visualizations:
   - Contour plots (as shown above)
   - Vector fields (quiver plots)
   - Line probes for profile analysis
   - Streamlines for flow visualization


In [12]:
# Example 1: Extract a 2D slice (XZ plane at a fixed Y index)
# This samples a cross-section through the wake

y_index = 36  # Can be 0 to 1324
variables_to_extract = [1, 3, 6]  # X, Z, U, V, W
output_file = f'./slice_xz_y{y_index}.csv'

print(f"\nExtracting XZ slice at Y index {y_index}...")
print("(This reads ~384M float values from the 28.9 GB file, extracting only the slice)")

slice_result = reader.extract_2d_slice(
    plane='XZ',
    index=y_index,
    var_indices=variables_to_extract,
    output_csv=output_file
)

print(f"\nSlice info:")
print(f"  Grid shape: {slice_result['grid_dims']}")
print(f"  Variables: {slice_result['variables']}")
print(f"  Points per variable: {len(slice_result['data'][slice_result['variables'][0]])}")


Extracting XZ slice at Y index 36...
(This reads ~384M float values from the 28.9 GB file, extracting only the slice)
Extracting XZ slice at index 36...
Grid dimensions: 1325 x 1325 = 1755625 points
Total points in full 3D grid: 1325 x 73 x 1325 = 128160625
Slice contains 1755625 points
  Reading variable 1 (X)...
  Reading variable 3 (Z)...
  Reading variable 6 (W)...
Extracted 3 variables
Writing to ./slice_xz_y36.csv...
Saved to ./slice_xz_y36.csv

Slice info:
  Grid shape: (1325, 1325)
  Variables: ['X', 'Z', 'W']
  Points per variable: 1755625


In [8]:
# Example 2: Load the slice CSV and create contour plots

# Read the CSV into a DataFrame
df = pd.read_csv(output_file)
print(f"Loaded CSV: {output_file}")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

# Reshape into 2D grids for contouring
# Grid is (I, J) = (1325, 73) for XY plane
I, J = reader.zone['I'], reader.zone['J']

# Create pivot table-like structure for each variable
X_2d = df['X'].values.reshape((J, I))
Y_2d = df['Y'].values.reshape((J, I))
U_2d = df['U'].values.reshape((J, I))
V_2d = df['V'].values.reshape((J, I))
W_2d = df['W'].values.reshape((J, I))

print(f"\nReshaped grids to 2D: ({J}, {I})")
print(f"  X range: [{X_2d.min():.3f}, {X_2d.max():.3f}]")
print(f"  Y range: [{Y_2d.min():.3f}, {Y_2d.max():.3f}]")
print(f"  U range: [{U_2d.min():.3f}, {U_2d.max():.3f}]")

Loaded CSV: ./slice_xy_z36.csv
Shape: (96725, 5)
Columns: ['X', 'Y', 'U', 'V', 'W']

First few rows:
          X        Y    U         V    W
0 -0.000016 -0.28544  0.0  0.000819  0.0
1  0.076353 -0.28544  0.0  0.000811  0.0
2  0.152040 -0.28544  0.0  0.000782  0.0
3  0.227057 -0.28544  0.0  0.000784  0.0
4  0.301412 -0.28544  0.0  0.000674  0.0

Reshaped grids to 2D: (73, 1325)
  X range: [-0.000, 32.136]
  Y range: [-0.285, 1.000]
  U range: [-0.066, 0.038]


No exact X match found for line probe extraction

--- Post-Processing Options Summary ---
1. Extract different 2D planes:
   - XY plane: reader.extract_2d_slice(plane='XY', index=z_idx)
   - XZ plane: reader.extract_2d_slice(plane='XZ', index=y_idx)
   - YZ plane: reader.extract_2d_slice(plane='YZ', index=x_idx)

2. Extract variables of interest:
   - Variables available: ['X', 'Y', 'Z', 'U', 'V', 'W', 'uu', 'vv', 'ww', 'uv', 'vw', 'uw', 'K', 'Nv']
   - Pass var_indices=[...] to extract_2d_slice()

3. Save to different formats:
   - CSV (default): output_csv='file.csv'
   - Convert to HDF5 or Zarr for efficient slicing

4. Create visualizations:
   - Contour plots (as shown above)
   - Vector fields (quiver plots)
   - Line probes for profile analysis
   - Streamlines for flow visualization


In [17]:
# Finding Grid Indices from Coordinates

print("=== Coordinate to Index Mapping ===\n")

# Method 1: Find nearest grid point to a target coordinate
print("Method 1: Find nearest point to coordinates (X=10, Y=0)")
mask = np.ones(len(df), dtype=bool)
mask &= np.abs(df['X'] - 10.0) < 1
mask &= np.abs(df['Y'] - 0.0) < 1
candidates = df[mask]
dist = (candidates['X'].values - 10.0)**2 + (candidates['Y'].values - 0.0)**2
nearest_idx = candidates.index[np.argmin(dist)]
nearest_point = df.loc[nearest_idx]
print(f"  Dataframe index: {nearest_idx}")
print(f"  Coordinates: X={nearest_point['X']:.4f}, Y={nearest_point['Y']:.4f}")
print(f"  Velocity: U={nearest_point['U']:.4f}, V={nearest_point['V']:.4f}\n")

# Method 2: Find all points on a vertical line (constant X)
print("Method 2: Find all points at X ≈ 10.0")
x_target = 10.0
line_indices = df[np.abs(df['X'] - x_target) < 0.1].index.tolist()
print(f"  Found {len(line_indices)} points on this vertical line")
print(f"  Y range on this line: [{df.loc[line_indices, 'Y'].min():.4f}, {df.loc[line_indices, 'Y'].max():.4f}]\n")

# Method 3: Get coordinate value ranges  
print("Method 3: Available coordinate ranges in this slice:")
print(f"  X: [{df['X'].min():.4f}, {df['X'].max():.4f}]  ({df['X'].nunique()} unique grid points)")
print(f"  Y: [{df['Y'].min():.4f}, {df['Y'].max():.4f}]  ({df['Y'].nunique()} unique grid points)")
if 'Z' in df.columns:
    print(f"  Z: [{df['Z'].min():.4f}, {df['Z'].max():.4f}]  ({df['Z'].nunique()} unique grid points)")
else:
    print(f"  Z: (not in this XZ slice)")

print("\n--- Common Use Cases ---")
print("1. Extract a specific row by index:")
print("   row_data = df.iloc[100]  # Get row 100")
print("   row_data = df.loc[29427]  # Get by index label")
print("\n2. Find nearest point to coordinate:")
print("   mask = (np.abs(df['X']-10) < 0.5) & (np.abs(df['Y']-0) < 0.5)")
print("   subset = df[mask]")
print("\n3. Extract vertical profile (constant X):")
print("   profile = df[np.abs(df['X']-10) < 0.1].sort_values('Y')")
print("\n4. Extract horizontal profile (constant Y):")
print("   profile = df[np.abs(df['Y']-0.1) < 0.01].sort_values('X')")

=== Coordinate to Index Mapping ===

Method 1: Find nearest point to coordinates (X=10, Y=0)
  Dataframe index: 29427
  Coordinates: X=9.9914, Y=0.0060
  Velocity: U=-0.0289, V=0.0000

Method 2: Find all points at X ≈ 10.0
  Found 730 points on this vertical line
  Y range on this line: [-0.2854, 1.0000]

Method 3: Available coordinate ranges in this slice:
  X: [-0.0001, 32.1359]  (43281 unique grid points)
  Y: [-0.2854, 1.0000]  (74 unique grid points)
  Z: (not in this XZ slice)

--- Common Use Cases ---
1. Extract a specific row by index:
   row_data = df.iloc[100]  # Get row 100
   row_data = df.loc[29427]  # Get by index label

2. Find nearest point to coordinate:
   mask = (np.abs(df['X']-10) < 0.5) & (np.abs(df['Y']-0) < 0.5)
   subset = df[mask]

3. Extract vertical profile (constant X):
   profile = df[np.abs(df['X']-10) < 0.1].sort_values('Y')

4. Extract horizontal profile (constant Y):
   profile = df[np.abs(df['Y']-0.1) < 0.01].sort_values('X')


## Wake data visualizations

https://www.mathworks.com/matlabcentral/fileexchange/66237-matlab-tool-to-read-tecplot-ascii-dat-tec2mat